---
jupyter: python3
#code-fold: true
execute:
  echo: false
  output: asis
lightbox: true
loop: true
---

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
from pathlib import Path
from collections import defaultdict

def query_WB(endpoint, query):
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.query().convert()

def get_data(results):
    return [
        {
            "url":           result["item"]["value"],
            "label":         result["itemLabel"]["value"],
            "description":   result["itemDescription"]["value"],
            "photo":         result.get("photo",{}).get("value",""),
            "roomLink":      result.get("parent",{}).get("value",""),
            "room":          result.get("parentLabel",{}).get("value",""),
            "type":          result["itemTypeLabel"]["value"],
            "creator":       result.get("creator",{}).get("value","-"),
        }
        for result in results["results"]["bindings"]
    ]

def group_data(data):
    grouped = defaultdict(lambda: {
        "label": None,
        "wikibase_url": None,
        "type": None,
        "description": None,
        "photos": [],
        "creator": None,       # optional: last creator seen for this artwork
        "room": None,          # optional: last room seen for this artwork
        "roomLink": None,      # optional: last roomLink seen for this artwork
    })
    for entry in data:
        key_art = entry["label"]
        grouped[key_art]["label"] = entry["label"]
        grouped[key_art]["wikibase_url"] = entry["url"]
        grouped[key_art]["type"] = entry["type"]
        grouped[key_art]["description"] = entry["description"]
        grouped[key_art]["creator"] = entry["creator"]
        grouped[key_art]["room"] = entry.get("room", "")
        grouped[key_art]["roomLink"] = entry.get("roomLink", "")
        grouped[key_art]["photos"].append({
            "photo": entry["photo"],
            "photographer": entry["creator"],
        })
    return grouped

def is_supported_image(path):
    ext = Path(path).suffix.lower()
    return ext in [".jpg", ".jpeg", ".png"]

def generate_output(grouped, castle_label):
    print(f""" 
# Liste (Malereien): {castle_label}\n
    """)
    for art in grouped.values():
        print(f"""
## {art["label"]}

Wikibase: [{art["label"]}]({art["wikibase_url"]})  
        """)
        for entry in art["photos"]:
            if not is_supported_image(entry["photo"]):
                continue
            print(f"""
![{art["label"]}. Foto: {entry["photographer"]}]({entry["photo"]}){{group="photos"}}

Foto: {entry["photographer"]}  
Objektart: {art["type"]}  
Raum: [{art["room"]}]({art["roomLink"]})

---

\\newpage
            """)



In [2]:
castle_ID = "Q68"

In [3]:
def make_book():

    endpoint = "https://query.kewl.org/sparql"

    query = f"""
PREFIX wd: <https://wikibase.kewl.org/entity/>
PREFIX wdt: <https://wikibase.kewl.org/prop/direct/>
PREFIX p: <https://wikibase.kewl.org/prop/>
PREFIX ps: <https://wikibase.kewl.org/prop/statement/>
PREFIX pq: <https://wikibase.kewl.org/prop/qualifier/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX bd: <http://www.bigdata.com/rdf#>


SELECT DISTINCT ?item ?itemLabel ?itemDescription ?itemTypeLabel ?photo ?creator ?parent ?parentLabel ?castleLabel

WHERE {{
  
  VALUES ?itemType {{ wd:Q6 }}
  
  ?item wdt:P3+ wd:{castle_ID} ;
        wdt:P1 ?itemType .
  OPTIONAL {{ ?item wdt:P3 ?parent . }}
  OPTIONAL {{
    ?item p:P6 ?statement .
    ?statement ps:P6 ?photo .
    OPTIONAL {{ ?statement pq:P11 ?creator. }}
  }}
  BIND(wd:{castle_ID} AS ?castle)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "de" }}
}}
ORDER BY ?item
    """

    results = query_WB(endpoint, query)
    castle = next((r["castleLabel"]["value"] for r in results["results"]["bindings"] if "castleLabel" in r))
    data = get_data(results)
    grouped = group_data(data)
    generate_output(grouped, castle)

make_book()

 
# Liste (Malereien): Stuttgart, Schloss Solitude

    

## Wanddekoration mit Blumengirlanden

Wikibase: [Wanddekoration mit Blumengirlanden](https://wikibase.kewl.org/entity/Q90)  
        

![Wanddekoration mit Blumengirlanden. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013272a.jpg){group="photos"}

Foto: Bunz, Achim  
Objektart: Malerei  
Raum: [Kabinett](https://wikibase.kewl.org/entity/Q89)

---

\newpage
            

![Wanddekoration mit Blumengirlanden. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013290a.jpg){group="photos"}

Foto: Bunz, Achim  
Objektart: Malerei  
Raum: [Kabinett](https://wikibase.kewl.org/entity/Q89)

---

\newpage
            

## Die Andacht

Wikibase: [Die Andacht](https://wikibase.kewl.org/entity/Q78)  
        

## Auferstehung Christi

Wikibase: [Auferstehung Christi](https://wikibase.kewl.org/entity/Q80)  
        

![Auferstehung Christi. Foto: Bunz, Achim](https://previous.bildindex.de/bilder/fmd10013274a.jpg){gr

In [4]:
# Query Link

# https://query.kewl.org/#PREFIX%20wd%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fentity%2F%3E%0APREFIX%20wdt%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2Fdirect%2F%3E%0APREFIX%20p%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2F%3E%0APREFIX%20ps%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2Fstatement%2F%3E%0APREFIX%20pq%3A%20%3Chttps%3A%2F%2Fwikibase.kewl.org%2Fprop%2Fqualifier%2F%3E%0APREFIX%20wikibase%3A%20%3Chttp%3A%2F%2Fwikiba.se%2Fontology%23%3E%0APREFIX%20rdfs%3A%20%3Chttp%3A%2F%2Fwww.w3.org%2F2000%2F01%2Frdf-schema%23%3E%0APREFIX%20bd%3A%20%3Chttp%3A%2F%2Fwww.bigdata.com%2Frdf%23%3E%0A%0A%0ASELECT%20DISTINCT%20%3Fitem%20%3FitemLabel%20%3FitemDescription%20%3FitemTypeLabel%20%3Fphoto%20%3Fcreator%20%3Fparent%20%3FparentLabel%20%3FcastleLabel%0A%0AWHERE%20%7B%0A%20%20%0A%20%20VALUES%20%3FitemType%20%7B%20wd%3AQ6%20%7D%0A%20%20%0A%20%20%3Fitem%20wdt%3AP3%2B%20wd%3AQ68%20%3B%0A%20%20%20%20%20%20%20%20wdt%3AP1%20%3FitemType%20.%0A%20%20OPTIONAL%20%7B%20%3Fitem%20wdt%3AP3%20%3Fparent%20.%20%7D%0A%20%20OPTIONAL%20%7B%0A%20%20%20%20%3Fitem%20p%3AP6%20%3Fstatement%20.%0A%20%20%20%20%3Fstatement%20ps%3AP6%20%3Fphoto%20.%0A%20%20%20%20OPTIONAL%20%7B%20%3Fstatement%20pq%3AP11%20%3Fcreator.%20%7D%0A%20%20%7D%0A%20%20BIND%28wd%3AQ68%20AS%20%3Fcastle%29%0A%20%20SERVICE%20wikibase%3Alabel%20%7B%20bd%3AserviceParam%20wikibase%3Alanguage%20%22de%22%20%7D%0A%7D%0AORDER%20BY%20%3Fitem